In [2]:
import requests
import pygsheets
import pandas as pd

In [ ]:
client = pygsheets.authorize(service_account_file=r'C:\Users\fumio\Documents\gcp-project-365616-6b0f83cca4c2.json')

ss = client.open_by_url('https://docs.google.com/spreadsheets/d/1H-otjPW382f7cUNvaC_p5BJCMw7nmlmuhu2JiNBx_Y0/edit?gid=195125089#gid=195125089')

ws = ss.worksheet_by_title('Props Iptu')

In [ ]:
df_in = ws.get_as_df()

In [ ]:
df = df_in[df_in['Extraído IPTUs'] == '']
df = df[df['Cond.'] != '']

In [ ]:
df.head(15)

In [ ]:
dfs = []

In [ ]:
def get_iptus(row: pd.Series, dfs: list):
    condominio = row['Condominio']

    body = {
        'pCdSetor': str(row['Setor']).rjust(3, '0'),
        'pCdQuadra': str(row['Quadra']).rjust(3, '0'),
        'pCdCondominio':str(row['Cond.']).rjust(2, '0'),
    }

    print(body)

    response = requests.post(
        url = 'https://geosampa.prefeitura.sp.gov.br/PaginasPublicas/_SBC.aspx/pesquisaLoteInfo',
        json = body,
    )

    response_json = response.json()

    if not len(response_json['d']):
        print(f'Algum problema aconteceu com a query {body}')
        return

    df = pd.json_normalize(response_json['d'])

    df['iptu'] = df.apply(lambda x: f"{x['cd_setor_fiscal']}.{x['cd_quadra_fiscal']}.{x['cd_lote']}-{x['cd_digito_sql']}", axis=1)
    df['condominio'] = condominio
    df = df[[
        'condominio',
        'iptu',
        'nm_logradouro_completo',
        'cd_numero_porta',
        'tx_complemento_endereco'
    ]]

    row['Extraído IPTUs'] = 'OK'

    dfs.append(df)

In [ ]:
df.apply(lambda x: get_iptus(x, dfs), axis=1)

In [3]:
# body = {'pCdSetor': '169', 'pCdQuadra': '112', 'pCdCondominio': '02'}

# response = requests.post(
#     url = 'https://geosampa.prefeitura.sp.gov.br/PaginasPublicas/_SBC.aspx/pesquisaLoteInfo',
#     json = body,
# )

# response_json = response.json()

In [ ]:
df_out = pd.concat(dfs)

In [ ]:
df_in['Extraído IPTUs'] = 'OK'

In [ ]:
ws.set_dataframe(
    df_in,
    'A1',
)

In [ ]:
ws = ss.worksheet_by_title('Automação GeoSampa')

In [ ]:
ws.insert_rows(
    1,
    len(df_out),
    df_out.values.tolist(),
)